In [ ]:
import torch
import pandas as pd
import wandb

from datasets import Dataset
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments
from trl import DataCollatorForCompletionOnlyLM

from rouge import Rouge
from tqdm import tqdm

## W&B Setting and Data Load

In [ ]:
# WANDB 세팅
wandb.init(project="SOLAR-Summarization", name="Qwen3-14B-Unsloth-v2")

# DataLoad
train_df = pd.read_csv('../Data/train_processed.csv')
dev_df = pd.read_csv('../Data/dev_processed.csv')

train_dataset = Dataset.from_pandas(train_df)
eval_dataset = Dataset.from_pandas(dev_df)

## Unsloth Model Load

In [ ]:
max_seq_length = 2048
dtype = None
load_in_4bit = True

print("💻 Unsloth 기반 Qwen-14B 모델 로딩 중...")

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3-14B",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit
)

In [ ]:
# Unsloth 전용 LoRA 설정 (속도 향상 그리고 메모리 절약 가능)
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 42,
    use_rslora = False,
    loftq_config = None,
)

## ChatML Prompt 적용

In [ ]:
SYSTEM_PROMPT = (
    "당신은 한국어 대화 요약 전문가입니다.\n"
    "대화에는 #Person1#, #Person2# 등의 화자 태그가 사용됩니다.\n"
    "요약할 때 이 화자 태그를 그대로 사용하여 누가 무엇을 했는지 명확히 구분해주세요.\n"
    "핵심 내용만 1~2문장으로 간결하게 요약하세요."
)

In [ ]:
def formatting_prompts_func(example):
    formatted_texts = []

    dialogues = example['dialogue'] if isinstance(example['dialogue'], list) else [example['dialogue']]
    summaries = example['summary'] if isinstance(example['summary'], list) else [example['summary']]

    for d, s in zip(dialogues, summaries):
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"아래 대화를 읽고 핵심 내용을 요약해주세요.\n화자 태그(#Person1# 등)를 유지하세요.\n\n[대화]\n{d}"},
            {"role": "assistant", "content": str(s)}
        ]

        # tokenizer 가 <|im_start|> 등의 특수 토큰을 알아서 붙여줌
        text = tokenizer.apply_chat_template(messages, tokenizer=False, add_generation_prompt=False)
        formatted_texts.append(text)

    return {"text": formatted_texts}

# Dataset 에 프롬프트 영구 각인
train_dataset = train_dataset.map(formatting_prompts_func, batched=True)
eval_dataset = eval_dataset.map(formatting_prompts_func, batched=True)

## FeedBack : Response-Only

In [ ]:
# Qwen 모델의 assistant 응답 시작 토큰
response_template = "<|im_start|>assistant\n"

collator = DataCollatorForCompletionOnlyLM(
    response_template=response_template,
    tokenizer=tokenizer
)

print("Response-Only Loss 세팅 완료")

## SFTTrainer 학습 세팅

In [ ]:
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = eval_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    data_collator = collator,
    dataset_num_proc = 2,
    args = TrainingArguments(
        output_dir = "outputs/qwen3_sft_v2",
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 32,
        warmup_ratio = 0.05,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 10,
        eval_strategy = "epoch",
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        seed = 42,
        report_to = "wandb"
    )
)

## Train

In [ ]:
trainer_stats = trainer.train()

# WB 종료
wandb.finish()

## Model Save

In [ ]:
best_model_path = "./Model_Save/qwen3_best_lora_v2"
model.save_pretrained(best_model_path)
tokenizer.save_pretrained(best_model_path)

## Rouge Score 

In [ ]:
# Unsloth : 추론에서도 추론 속도를 향상 시킬 수 있음
FastLanguageModel.for_inference(model)
# 커널 껐다 켰다 작업이 반복 될 수 있으니 경로 하드코딩
dev_df = pd.read_csv('../Data/dev_processed.csv')

print("검증 데이터로 로컬 Rouge 점수를 측정합니다.")

predicted_summaries = []
actual_summaries = dev_df['summary'].tolist()

for dialogue in tqdm(dev_df['dialogue']):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"아래 대화를 읽고 핵심 내용을 요약해주세요.\n화자 태그(#Person1# 등)를 유지하세요.\n\n[대화]\n{dialogue}"}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=100,
            repetition_penalty=1.2,
            temperatuire=0.1,
            pad_token_id=tokenizer.eos_token_id
        )

    # 프롬프트 찌꺼기를 안전하게 자르기
    # 모델이 뱉어낸 전체 토큰 길이에서, 우리가 넣었던 '질문 토큰 길이' 이후 부터만 잘라서 디코딩
    input_length = inputs['inputs_ids'].shape[1]
    generated_tokens = outputs[0][input_length:]
    summary_only = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

    # 예외처리 : 모델이 빈 칸을 뱉으면 . 으로 대체
    if len(summary_only) == 0:
        summary_only = "."
    
    predicted_summaries.append(summary_only)


In [ ]:
## ROUGE 채점
rouge = Rouge()
scores = rouge.get_scores(predicted_summaries, actual_summaries, avg=True)

# 리더보드 제출용 환산 점수
final_result = (scores['rouge-1']['f'] + scores['rouge-2']['f'] + scores['rouge-l']['f']) / 3.0 * 100.0

print("\n🏆 [로컬 모의고사 ROUGE 점수 결과] 🏆")
print(f"ROUGE-1: {scores['rouge-1']['f']:.4f}")
print(f"ROUGE-2: {scores['rouge-2']['f']:.4f}")
print(f"ROUGE-L: {scores['rouge-l']['f']:.4f}")
print(f"Final Result: {final_result:.2f}")

## Submission CSV

In [ ]:
test_df = pd.read_csv('../Data/test_processed.csv')
sample_submission = pd.read_csv('../Data/sample_submission.csv')

predicted_test_summaries = []

for dialogue in tqdm(test_df['dialogue']):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"아래 대화를 읽고 핵심 내용을 요약해주세요.\n화자 태그(#Person1# 등)를 유지하세요.\n\n[대화]\n{dialogue}"}
    ]

    text = tokenizer.apply_chat_template(messages, tokenizer=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=100,
            repetition_penalty=1.2,
            temperature=0.1,
            pad_token_id=tokenizer.eos_token_id
        )

    input_length = inputs['input_ids'].shape[1]
    generated_tokens = outputs[0][input_length:]
    summary_only = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

    predicted_test_summaries.append(summary_only)

In [ ]:
# Submission.csv 저장
sample_submission['summary'] = predicted_test_summaries
submission_path = '../Data/qwen3_submission_v1.csv'
sample_submission.to_csv(submission_path, index=False, encoding='utf-8-sig')

print(f"리더보드 제출용 파일이 생성되었습니다: {submission_path}")